# Gene Program Database Creation for MNC

This notebook createsa "gene program database" file (gpdb_tf.csv) for running Tripso, where column names are gene program names and the values are the genes in the gene program. Here we start from a selection of manually curated gene programs of intetest (gp_nw_all.csv), filters gene programs based on their expression patterns in the MNC dataset, and ensures that selected gene programs have sufficient gene coverage across cells. Empirically, we find that below 5 GP genes expressed per cell on average, Tripso does not learn meaningful representations. 

**Inputs:**
- `data/processed/mnc.h5ad`: h5ad object with gene epxression data
- `gp_nw_all.csv`: Manually curated set of gene programs

**Outputs:**
- `gpdb_tf.csv`: Filtered gene program database for Tripso model training

**Purpose:** Quality control step to ensure gene programs are relevant for the dataset of interest before tokenization and model training.

In [1]:
import pandas as pd
import scanpy as sc
import numpy as np
import os

## MNC

In [2]:
mnc = sc.read_h5ad('data/processed/mnc.h5ad')
mnc

AnnData object with n_obs × n_vars = 79583 × 36601
    obs: 'runid_mrna_sample', 'sorting', 'biological_replicate_labID', 'age', 'sex', 'tissue', 'age_general', 'phase', 'cell_type', 'donor_tissue', 'donor', 'source', 'tissue_source', 'n_counts', 'study'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'isCC', 'mean', 'std'
    varm: 'PCs'
    layers: 'counts', 'log_counts'

In [3]:
mnc.var.head()

,gene_ids,feature_types,mt,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,highly_variable,highly_variable_rank,means,variances,variances_norm,highly_variable_nbatches,isCC,mean,std
MIR1302-2HG,ENSG00000243485,Gene Expression,False,3,0.000037,0.000037,99.996287,3.0,1.386294,False,NaN,0.000037,0.000037,0.199982,0,False,0.000018,0.003655
FAM138A,ENSG00000237613,Gene Expression,False,0,0.000000,0.000000,100.000000,0.0,0.000000,False,NaN,0.000000,0.000000,0.000000,0,False,0.000000,1.000000
OR4F5,ENSG00000186092,Gene Expression,False,0,0.000000,0.000000,100.000000,0.0,0.000000,False,NaN,0.000000,0.000000,0.000000,0,False,0.000000,1.000000
AL627309.1,ENSG00000238009,Gene Expression,False,131,0.001621,0.001620,99.837859,131.0,4.882802,False,NaN,0.001621,0.001619,0.838603,0,False,0.000856,0.025543
AL627309.3,ENSG00000239945,Gene Expression,False,18,0.000223,0.000223,99.977721,18.0,2.944439,False,NaN,0.000223,0.000223,0.652627,0,False,0.000120,0.010504


## For each GP, check average number of genes per cell?
&rarr; from this can filter GP with low number of genes

In [5]:
import numpy as np

In [6]:
gpdb = pd.read_csv('gp_nw_all.csv')

In [9]:
gp_to_drop = []

for gp in gpdb.columns:
    print(f"Processing gene group: {gp}")  # Print the name of the current gene group
    
    genes = gpdb[gp].dropna().values  # Get the list of non-null gene names in the current group
        
    # Subset adata to only include genes that are in the current list of genes
    bdata = mnc[:, mnc.var.index.isin(genes)]  
    
    # Calculate the number of genes with non-zero expression in each cell
    non_zero_counts = (bdata.X > 0).sum(axis=1)  # Sum of non-zero values for each row (cell)
    
    # Calculate the average number of genes with non-zero expression per cell
    average_non_zero_genes = non_zero_counts.mean()  
    
    print(f"    Average number of genes with non-zero expression per cell: {average_non_zero_genes:.2f}")
    print('')
    
    if average_non_zero_genes < 5:
        gp_to_drop.append(gp)
        

Processing gene group: GP_USF1
    Average number of genes with non-zero expression per cell: 33.32

Processing gene group: GP_NFE2L2
    Average number of genes with non-zero expression per cell: 42.21

Processing gene group: GP_RUNX1
    Average number of genes with non-zero expression per cell: 31.36

Processing gene group: GP_FOXO3
    Average number of genes with non-zero expression per cell: 35.58

Processing gene group: GP_MYB
    Average number of genes with non-zero expression per cell: 36.04

Processing gene group: GP_E2F4
    Average number of genes with non-zero expression per cell: 45.27

Processing gene group: GP_IRF1
    Average number of genes with non-zero expression per cell: 22.61

Processing gene group: GP_GATA1
    Average number of genes with non-zero expression per cell: 28.48

Processing gene group: GP_CTCF
    Average number of genes with non-zero expression per cell: 23.44

Processing gene group: GP_MYCN
    Average number of genes with non-zero expression per

In [10]:
gpdb = gpdb.drop(columns = gp_to_drop)

In [11]:
gpdb.shape

(227, 33)

In [12]:
gpdb.to_csv('gpdb_tf.csv', index = False)